In [1]:
# 📦 Required installs
# !pip install langchain langchain-community langchain-classic langchain-text-splitters faiss-cpu python-dotenv
# !pip install langchain-google-genai
# !pip install tenacity

import os
from dotenv import load_dotenv
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_google_genai._common import GoogleGenerativeAIError

# 2️⃣ Load API keys
load_dotenv(".env")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")

# text-embedding-004 was deprecated Jan 14, 2026 — use gemini-embedding-001
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", max_retries=5)

# gemini-1.5-flash / gemini-2.5-flash are deprecated/shutting down — use current stable 3.x line.
# Run genai.list_models() against your key first if this 404s again — Google retires models fast.
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

# 1. Load the text file
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 2. Split the text into chunks
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(raw_text)
documents = [Document(page_content=chunk) for chunk in chunks]

# 3. Create vector store — batched + retried, since Google's embed_content endpoint is flaky
@retry(
    wait=wait_exponential(multiplier=1, min=2, max=30),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(GoogleGenerativeAIError),
    reraise=True,
)
def _embed_batch(store, batch, embedding):
    if store is None:
        return FAISS.from_documents(batch, embedding=embedding)
    store.add_documents(batch)
    return store


def build_vectorstore(all_documents, embedding, batch_size=10):
    store = None
    for i in range(0, len(all_documents), batch_size):
        batch = all_documents[i:i + batch_size]
        store = _embed_batch(store, batch, embedding)
    return store


vectorstore = build_vectorstore(documents, embedding, batch_size=10)
print(f"Indexed {len(documents)} chunks.")

C:\Users\Manikandan\AppData\Local\Temp\ipykernel_14368\3170686851.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Indexed 1 chunks.


In [2]:
#Get retrieves --> it does not generate
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Compressor using LLM
compressor = LLMChainExtractor.from_llm(llm)

# Create compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)

# Run retrieval
print("🔹 Contextual Compression Results:")
results = compression_retriever.invoke("Who created LangChain?")
for doc in results:
    print("-", doc.page_content)

🔹 Contextual Compression Results:
- Harrison Chase


In [4]:
#Integration with conversational RAG — replaces ConversationalRetrievalChain
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Step 1: history-aware retriever (rewrites follow-up questions using chat history)
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and the latest user question, "
               "formulate a standalone question which can be understood "
               "without the chat history. Do NOT answer the question, "
               "just reformulate it if needed and otherwise return it as is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(
    llm, compression_retriever, contextualize_q_prompt
)

# Step 2: answer generation, stuffing retrieved docs into the prompt
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's questions based on the below context:\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Step 3: combine into the full RAG chain
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# Step 4: memory — replaces ConversationBufferMemory
store = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# 🧪 Ask a few questions
print("🔹 ConversationalRetrievalChain (migrated):")
config = {"configurable": {"session_id": "session-1"}}
print(conversational_rag_chain.invoke({"input": "What is LangChain?"}, config=config)["answer"])
print(conversational_rag_chain.invoke({"input": "Who created it?"}, config=config)["answer"])

C:\Users\Manikandan\anaconda3\envs\langchain\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


🔹 ConversationalRetrievalChain (migrated):
Based on the provided context, LangChain is a framework for developing applications powered by large language models (LLMs).


GoogleGenerativeAIError: Error embedding content: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

In [ ]:
# !pip install google-genai
from google import genai
import os
from dotenv import load_dotenv

load_dotenv(".env")
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

for model in client.models.list():
    print(model.name)

In [5]:
#Integration into Agent (with Tool)
from langchain.agents import create_agent
from langchain_core.tools import StructuredTool

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    result = conversational_rag_chain.invoke(
        {"input": question},
        config={"configurable": {"session_id": "agent-session"}},
    )
    return result["answer"]


# Structured Tool
rag_tool = StructuredTool.from_function(
    name="RAG_Tool",
    description="Answer LangChain-related questions with context.",
    func=rag_tool_fn,
)

# Agent — create_agent runs on LangGraph under the hood and handles the tool-calling loop
agent = create_agent(
    model=llm,
    tools=[rag_tool],
    system_prompt="You are a helpful assistant. Use the RAG_Tool for LangChain-related questions.",
)

# 🧪 Ask via agent
print("\n🔹 Agent Conversation:")
result1 = agent.invoke({"messages": [{"role": "user", "content": "What is LangChain?"}]})
print(result1["messages"][-1].content)

result2 = agent.invoke({"messages": [{"role": "user", "content": "Who created it?"}]})
print(result2["messages"][-1].content)


🔹 Agent Conversation:
[{'type': 'text', 'text': 'LangChain is a popular open-source framework designed to simplify the creation of applications powered by large language models (LLMs). \n\nIt provides developers with the tools, components, and integrations needed to build LLM-powered applications, such as chatbots, question-answering systems, and agents, by making it easier to connect these models to external data sources and other software tools.', 'extras': {'signature': 'ErwCCrkCARFNMg8LTcg1zMBTlF19Af6upedH+9dqr5fy9+vv5ChcEtMOk6zwFWHxa2AaEf5yHTDlk7fwVIf+58vN8XFdMPtPXXSHMCzOq9KPCWJYwAmdNUcXAi7/l9dWmDw+k3hiy1p+rp0OXgyrjJT6QfoQ2HZkBcMPDfg63h7rUUOGJAUrUtH5jexvqVthZnB06tKgySAIuIcVzpChTXYZzBTu77Yz8dfVC5RrFZXKE7x++gDvyTwOtjSd2sMmgKZlFpqnFyiIFjLp5o1IcD2uP94vA0z8Ie6EGy5UgUebkF0m22BZCvejBdZM46aqZG54lkCoaxUj/BoV2+KaSmR5/EWETNWqKphSW7hsG09IwL+cw3M7ltvZkDDkS8kEkJ2CU8CfHuQBT1AeY0vHKzD6C2O5ndAaINmPHK3SOg=='}}]


GoogleGenerativeAIError: Error embedding content: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

In [6]:
#multiQuery
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Answer-generation chain (replaces RetrievalQA.from_chain_type)
qa_prompt_multi = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's question based on the below context:\n\n{context}"),
    ("human", "{input}"),
])
combine_docs_chain = create_stuff_documents_chain(llm, qa_prompt_multi)
rag_multi = create_retrieval_chain(multi_query_retriever, combine_docs_chain)

# 🧪 Ask question
print("\n🔹 RAG Pipeline (MultiQueryRetriever):")
res = rag_multi.invoke({"input": "Tell me about LangChain creator and features."})

print("Answer:", res["answer"])
print("\nSources:")
for doc in res["context"]:
    print("-", doc.page_content[:200])


🔹 RAG Pipeline (MultiQueryRetriever):
Answer: Based on the provided context, here is the information about LangChain's creator and features:

*   **Creator:** LangChain was created by **Harrison Chase** and was first released in October 2022.
*   **Features:** LangChain is a framework for developing applications powered by large language models (LLMs). Its features and capabilities include:
    *   Tools for chaining together LLM calls.
    *   Integrating retrieval-augmented generation (RAG).
    *   Connecting to vector stores.
    *   Building agents that can use external tools.
    *   Enabling the creation of LLM-powered applications such as chatbots, question-answering systems, and autonomous agents.

Sources:
- LangChain is a framework for developing applications powered by large language models (LLMs).
It provides tools for chaining together LLM calls, integrating retrieval-augmented generation (RAG),
conne
